## Read Silver

In [0]:
from pyspark.sql.functions import (
    sum, 
    col, 
    when, 
    round
)

from pyspark.sql.window import Window

silver_table = "ev_registration.silver.silver_vehicle_registrations"

gold_manufacturer_source_df = spark.read.table(silver_table)

print("silver records:", gold_manufacturer_source_df.count())

display(gold_manufacturer_source_df.limit(10))


silver records: 499995


registration_year,maker,state,rto_code,rto,vehicle_category,vehicle_model,fuel,vehicle_class,sale_type,vehicle_count,ingestion_timestamp,registration_month,registration_month_number,is_ev,ev_type
2023,HYUNDAI MOTOR INDIA LTD,Punjab,PB19,RTO BARNALA,LIGHT MOTOR VEHICLE,CRETA 1.5 CRDI MT EX,DIESEL,Motor Car,Resale,1,2026-10-06T17:16:18.405Z,Mar,3,0,NON_EV
2022,TVS MOTOR COMPANY LTD,Odisha,OR26,NUAPADA RTO,TWO WHEELER(NT),TVS RADEON,PETROL,M-Cycle/Scooter,New,3,2026-10-06T17:16:18.405Z,Mar,3,0,NON_EV
2020,TOYOTA KIRLOSKAR MOTOR PVT LTD,Karnataka,KA63,DHARWAD EAST RTO,LIGHT MOTOR VEHICLE,TOYOTA URBAN CRUISER (HIGH MT),PETROL,Motor Car,New,2,2026-10-06T17:16:18.405Z,Dec,12,0,NON_EV
2022,MARUTI SUZUKI INDIA LTD,Chhattisgarh,CG14,Jashpur DTO,LIGHT MOTOR VEHICLE,SWIFT LXI,PETROL,Motor Car,New,1,2026-10-06T17:16:18.405Z,Sep,9,0,NON_EV
2023,TATA MOTORS PASSENGER VEHICLES LTD,Karnataka,KA51,ELECTRONIC CITY RTO,LIGHT MOTOR VEHICLE,HARRIER XZ+ 2LBS6PH2 #DRK,DIESEL,Motor Car,New,1,2026-10-06T17:16:18.405Z,Sep,9,0,NON_EV
2021,HERO ELECTRIC VEHICLES PVT. LTD,Rajasthan,RJ14,JAIPUR (FIRST) RTO,TWO WHEELER(NT),OPTIMA HS 500 ER,ELECTRIC(BOV),M-Cycle/Scooter,Resale,1,2026-10-06T17:16:18.405Z,Jan,1,1,BATTERY_EV
2021,BAJAJ AUTO LTD,Maharashtra,MH40,NAGPUR (RURAL),THREE WHEELER(T),BAJAJ MAXIMA C,DIESEL,Three Wheeler (Goods),Resale,1,2026-10-06T17:16:18.405Z,Jan,1,0,NON_EV
2020,SUZUKI MOTORCYCLE INDIA PVT LTD,Maharashtra,MH28,BULDHANA,TWO WHEELER(NT),ACCESS 125 DISC(ALLOY WHEEL),PETROL,M-Cycle/Scooter,Resale,1,2026-10-06T17:16:18.405Z,Oct,10,0,NON_EV
2023,HONDA MOTORCYCLE AND SCOOTER INDIA (P) LTD,Bihar,BR39,KATIHAR,TWO WHEELER(NT),ACTIVA SMART,PETROL,M-Cycle/Scooter,New,1,2026-10-06T17:16:18.405Z,Sep,9,0,NON_EV
2023,TVS MOTOR COMPANY LTD,Arunachal Pradesh,AR01,ITANAGAR CAPITAL REGION,TWO WHEELER(NT),TVS NTORQ 125 XT,PETROL,M-Cycle/Scooter,New,14,2026-10-06T17:16:18.405Z,Oct,10,0,NON_EV


## Aggregate by Manufacturer

In [0]:
gold_manufacturer_df = (
    gold_manufacturer_source_df
    .groupBy("maker")
    .agg(
        sum("vehicle_count").alias("total_registrations"),
        sum(when(col("is_ev") == 1, col("vehicle_count")).otherwise(0)).alias("ev_registrations")
    )
)

display(
    gold_manufacturer_df
    .orderBy(col("ev_registrations").desc())
)

maker,total_registrations,ev_registrations
OLA ELECTRIC TECHNOLOGIES PVT LTD,19244,19244
TVS MOTOR COMPANY LTD,340706,15297
BAJAJ AUTO LTD,260276,12845
ATHER ENERGY LTD,10807,10807
OKINAWA AUTOTECH PVT LTD,4108,4108
YC ELECTRIC VEHICLE,4068,4068
HERO ELECTRIC VEHICLES PVT. LTD,3502,3502
TATA PASSENGER ELECTRIC MOBILITY LTD,3454,3454
HERO MOTOCORP LTD,666843,3377
MAHINDRA LAST MILE MOBILITY LTD,4363,3154


## Calculate Non-EV Registrations

In [0]:
gold_manufacturer_df = gold_manufacturer_df.withColumn(
    "non_ev_registrations",
    col("total_registrations") - col("ev_registrations")
)

display(
    gold_manufacturer_df
    .orderBy(col("ev_registrations").desc()
))


maker,total_registrations,ev_registrations,non_ev_registrations
OLA ELECTRIC TECHNOLOGIES PVT LTD,19244,19244,0
TVS MOTOR COMPANY LTD,340706,15297,325409
BAJAJ AUTO LTD,260276,12845,247431
ATHER ENERGY LTD,10807,10807,0
OKINAWA AUTOTECH PVT LTD,4108,4108,0
YC ELECTRIC VEHICLE,4068,4068,0
HERO ELECTRIC VEHICLES PVT. LTD,3502,3502,0
TATA PASSENGER ELECTRIC MOBILITY LTD,3454,3454,0
HERO MOTOCORP LTD,666843,3377,663466
MAHINDRA LAST MILE MOBILITY LTD,4363,3154,1209


## Calculate EV Penetration %

In [0]:
gold_manufacturer_df = gold_manufacturer_df.withColumn(
    "ev_penetration_pct",
    round(
        when(
            col("total_registrations") > 0,
            (
                col("ev_registrations") / 
                col("total_registrations")
            ) * 100 
        ).otherwise(0),
        2
    )
)

display(
    gold_manufacturer_df
    .orderBy(col("ev_penetration_pct").desc())
)

maker,total_registrations,ev_registrations,non_ev_registrations,ev_penetration_pct
EVTRIC MOTORS PVT LTD,26,26,0,100.0
LECTRIX E VEHICLES PVT LTD,653,653,0,100.0
CHARUVIKRAM AUTOMOBILE PVT LTD,1,1,0,100.0
Kiran Automobile Industry,10,10,0,100.0
ECOBIT INTERNATIONAL(IMPORTER: JHEV MOTORS),1,1,0,100.0
VIMAL UNIVERSAL TRADE PVT LTD,58,58,0,100.0
ZHEJIANG TIANYING (IMPORTER: NEWTRON ELECTRIC),6,6,0,100.0
SIRF INDIA VEHICLES PVT LTD,10,10,0,100.0
A K AUTTO ELECTRICAL,1,1,0,100.0
KERALA AUTOMOBILES LIMITED,1,1,0,100.0


## Calculate EV Market Share %

In [0]:
total_ev_window = Window.rowsBetween(
    Window.unboundedPreceding,
    Window.unboundedFollowing
)

gold_manufacturer_df = gold_manufacturer_df.withColumn(
    "total_ev_registrations",
    sum("ev_registrations").over(total_ev_window)
)
    
gold_manufacturer_df = gold_manufacturer_df.withColumn(
    "ev_market_share_pct",
    round(
        when(
            col("total_ev_registrations") > 0,
            (
                col("ev_registrations") / 
                col("total_ev_registrations")
            ) * 100
        ).otherwise(0),
        2
    )
)

gold_manufacturer_df = gold_manufacturer_df.drop("total_ev_registrations")

display(
    gold_manufacturer_df
    .orderBy(col("ev_market_share_pct").desc())
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


maker,total_registrations,ev_registrations,non_ev_registrations,ev_penetration_pct,ev_market_share_pct
OLA ELECTRIC TECHNOLOGIES PVT LTD,19244,19244,0,100.0,13.38
TVS MOTOR COMPANY LTD,340706,15297,325409,4.49,10.63
BAJAJ AUTO LTD,260276,12845,247431,4.94,8.93
ATHER ENERGY LTD,10807,10807,0,100.0,7.51
OKINAWA AUTOTECH PVT LTD,4108,4108,0,100.0,2.86
YC ELECTRIC VEHICLE,4068,4068,0,100.0,2.83
HERO ELECTRIC VEHICLES PVT. LTD,3502,3502,0,100.0,2.43
TATA PASSENGER ELECTRIC MOBILITY LTD,3454,3454,0,100.0,2.4
HERO MOTOCORP LTD,666843,3377,663466,0.51,2.35
MAHINDRA LAST MILE MOBILITY LTD,4363,3154,1209,72.29,2.19


## Final Column Order

In [0]:
gold_manufacturer_df = gold_manufacturer_df.select(
    "maker",
    "total_registrations",
    "ev_registrations",
    "non_ev_registrations",
    "ev_penetration_pct",
    "ev_market_share_pct"
)

gold_manufacturer_df.printSchema()

display(
    gold_manufacturer_df
    .orderBy(col("ev_registrations").desc())
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


root
 |-- maker: string (nullable = true)
 |-- total_registrations: long (nullable = true)
 |-- ev_registrations: long (nullable = true)
 |-- non_ev_registrations: long (nullable = true)
 |-- ev_penetration_pct: double (nullable = true)
 |-- ev_market_share_pct: double (nullable = true)



maker,total_registrations,ev_registrations,non_ev_registrations,ev_penetration_pct,ev_market_share_pct
OLA ELECTRIC TECHNOLOGIES PVT LTD,19244,19244,0,100.0,13.38
TVS MOTOR COMPANY LTD,340706,15297,325409,4.49,10.63
BAJAJ AUTO LTD,260276,12845,247431,4.94,8.93
ATHER ENERGY LTD,10807,10807,0,100.0,7.51
OKINAWA AUTOTECH PVT LTD,4108,4108,0,100.0,2.86
YC ELECTRIC VEHICLE,4068,4068,0,100.0,2.83
HERO ELECTRIC VEHICLES PVT. LTD,3502,3502,0,100.0,2.43
TATA PASSENGER ELECTRIC MOBILITY LTD,3454,3454,0,100.0,2.4
HERO MOTOCORP LTD,666843,3377,663466,0.51,2.35
MAHINDRA LAST MILE MOBILITY LTD,4363,3154,1209,72.29,2.19


## Validate Totals

In [0]:
gold_manufacturer_df.selectExpr(
    "sum(total_registrations) as total_registrations",
    "sum(ev_registrations) as total_ev_registrations",
    "sum(non_ev_registrations) as total_non_ev_registrations"
).show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-------------------+----------------------+--------------------------+
|total_registrations|total_ev_registrations|total_non_ev_registrations|
+-------------------+----------------------+--------------------------+
|            2806654|                143873|                   2662781|
+-------------------+----------------------+--------------------------+



## Check Manufacturer Count

In [0]:
print(
    "Number of manufacturers:",
    gold_manufacturer_df.select("maker").distinct().count()
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Number of manufacturers: 1244


## Write Gold Table

In [0]:
gold_manufacturer_table = "ev_registration.gold.gold_ev_manufacturer"

gold_manufacturer_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(gold_manufacturer_table)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


## Verify Gold Table

In [0]:
final_gold_manufacturer_df = spark.read.table(
    gold_manufacturer_table
)

print(
    "Gold manufacturer records:",
    final_gold_manufacturer_df.count()
)

final_gold_manufacturer_df.printSchema()

display(
    final_gold_manufacturer_df
    .orderBy(col("ev_registrations").desc())
)

Gold manufacturer records: 1244
root
 |-- maker: string (nullable = true)
 |-- total_registrations: long (nullable = true)
 |-- ev_registrations: long (nullable = true)
 |-- non_ev_registrations: long (nullable = true)
 |-- ev_penetration_pct: double (nullable = true)
 |-- ev_market_share_pct: double (nullable = true)



maker,total_registrations,ev_registrations,non_ev_registrations,ev_penetration_pct,ev_market_share_pct
OLA ELECTRIC TECHNOLOGIES PVT LTD,19244,19244,0,100.0,13.38
TVS MOTOR COMPANY LTD,340706,15297,325409,4.49,10.63
BAJAJ AUTO LTD,260276,12845,247431,4.94,8.93
ATHER ENERGY LTD,10807,10807,0,100.0,7.51
OKINAWA AUTOTECH PVT LTD,4108,4108,0,100.0,2.86
YC ELECTRIC VEHICLE,4068,4068,0,100.0,2.83
HERO ELECTRIC VEHICLES PVT. LTD,3502,3502,0,100.0,2.43
TATA PASSENGER ELECTRIC MOBILITY LTD,3454,3454,0,100.0,2.4
HERO MOTOCORP LTD,666843,3377,663466,0.51,2.35
MAHINDRA LAST MILE MOBILITY LTD,4363,3154,1209,72.29,2.19


In [0]:
final_gold_manufacturer_df.select("maker", "ev_registrations", "ev_market_share_pct").orderBy(col("ev_market_share_pct").desc()).show(10,False)

+------------------------------------+----------------+-------------------+
|maker                               |ev_registrations|ev_market_share_pct|
+------------------------------------+----------------+-------------------+
|OLA ELECTRIC TECHNOLOGIES PVT LTD   |19244           |13.38              |
|TVS MOTOR COMPANY LTD               |15297           |10.63              |
|BAJAJ AUTO LTD                      |12845           |8.93               |
|ATHER ENERGY LTD                    |10807           |7.51               |
|OKINAWA AUTOTECH PVT LTD            |4108            |2.86               |
|YC ELECTRIC VEHICLE                 |4068            |2.83               |
|HERO ELECTRIC VEHICLES PVT. LTD     |3502            |2.43               |
|TATA PASSENGER ELECTRIC MOBILITY LTD|3454            |2.4                |
|HERO MOTOCORP LTD                   |3377            |2.35               |
|MAHINDRA LAST MILE MOBILITY LTD     |3154            |2.19               |
+-----------